# Hikâye Oyuncağı: Türkçe ses modeli eğitimi (Colab)

**Çalışma zamanı → Çalışma zamanı türünü değiştir → GPU: A100 (yoksa L4)** seçin. Sonra hücreleri sırayla çalıştırın.

- Bütün kayıtlar Google Drive'daki `hikaye_ses/` klasörüne yazılır.
- Oturum koparsa, tekrar bağlanıp **bütün hücreleri baştan çalıştırın**: veri Drive'dan geri gelir, eğitim kaldığı yerden sürer.
- Aşamalar: veri (ilk seferde ~1 saat) → akustik → vocoder → GTA ince ayar. Bitmiş aşama atlanır.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/hikaye_ses'
import os; os.makedirs(DRIVE + '/ses_calisma', exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf hikaye-oyuncak
!git clone -q --depth 1 -b claude/wonderful-pasteur-x7seiq https://github.com/yusuf7855/hikaye-oyuncak.git
%cd /content/hikaye-oyuncak
!pip -q install edge-tts soundfile
# eğitim kayıtları doğrudan Drive'a
!ln -sfn {DRIVE}/ses_calisma ses_calisma
!git log --oneline -1

## 1. Veri
Drive'da `ses_veri.tar` varsa oradan açılır (1-2 dk). Yoksa 25 bin cümle Emel sesiyle indirilir, hazırlanır ve Drive'a arşivlenir.

In [ ]:
import os, subprocess
TAR = DRIVE + '/ses_veri.tar'
if os.path.exists(TAR):
    !cp {TAR} /content/ses_veri.tar && tar xf /content/ses_veri.tar && rm /content/ses_veri.tar
else:
    # indirme; eksik kalırsa tekrar dener
    for deneme in range(5):
        !python ses/veri_uret.py --cikti ses_veri --es 16 2>&1 | tail -2
        if len(os.listdir('ses_veri/mp3')) >= 24900: break
    !python ses/hazirla.py --veri ses_veri --is {os.cpu_count()}
    !tar cf /content/ses_veri.tar ses_veri/hazir ses_veri/egitim.txt ses_veri/dogrulama.txt ses_veri/uzunluk.tsv ses_veri/metin.tsv
    !cp /content/ses_veri.tar {TAR}
!wc -l ses_veri/egitim.txt ses_veri/dogrulama.txt

## 2. Akustik model (metin → mel)
Örnekler: Drive `hikaye_ses/ses_calisma/akustik/ornek/` (vocoder yokken Griffin-Lim ile, kaba).

In [ ]:
import os
IS = min(8, os.cpu_count())
!python ses/egit_akustik.py --is {IS} 2>&1 | tee -a ses_calisma/akustik.log | grep -E "adım [0-9]+000 |==|devam|cihaz|bitti|Error|error" 

## 3. Vocoder (mel → ses)
Örnekler: `ses_calisma/vocoder/ornek/` (`t*.wav` metinden, `v*.wav` gerçek mel'den).

In [ ]:
!python ses/egit_vocoder.py --is {IS} 2>&1 | tee -a ses_calisma/vocoder.log | grep -E "adım [0-9]+000 |==|devam|cihaz|bitti|Error|error" 

## 4. GTA ince ayar (vocoder akustiğin kendi çıktısıyla)

In [ ]:
!python ses/egit_vocoder.py --gta --cikti ses_calisma/vocoder_gta --adim 100000 --sadece-mel 0 --lr 1e-4 \
    --baslangic ses_calisma/vocoder/son.pt --is {IS} 2>&1 | tee -a ses_calisma/gta.log | grep -E "adım [0-9]+000 |==|devam|başlangıç|cihaz|bitti|Error|error" 

## Dinle

In [ ]:
from IPython.display import Audio, display
v = 'ses_calisma/vocoder_gta/son.pt' if os.path.exists('ses_calisma/vocoder_gta/son.pt') else 'ses_calisma/vocoder/son.pt'
!python ses/sentezle.py --vocoder {v} --cikti /content/deneme.wav --kuant
display(Audio('/content/deneme.wav'))